> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の英語原文を日本語に翻訳したものです（Apache License 2.0）。コードの動作は原文から変更していません。

# 例 #2 - ファイルの操作

まず、データファイルが揃っているか確認しましょう。`os` ライブラリがとても役に立ちます。

In [ ]:
from os import listdir
from os.path import join

listdir('.')

In [ ]:
listdir('../data')

`pandas` に組み込まれている `read_csv` 関数を使います。

In [ ]:
import pandas as pd
pd.read_csv?

`pandas` には他にどんな「read」系の関数があるでしょうか？

In [ ]:
import re
regex = re.compile(r'read')
list(filter(regex.match, dir(pd)))

ここでは `read_csv` に絞って、[openflights.org](openflights.org) のデータを調べてみましょう。

In [ ]:
routes = pd.read_csv(join('..', 'data', 'routes.csv'))
airports = pd.read_csv(join('..', 'data', 'airports.csv'))

In [ ]:
routes.head()

In [ ]:
routes.info()

欠損しているデータ点がいくつかあるようです……それがどんな状態なのか、可視化できるでしょうか？

In [ ]:
import missingno as msno

`missingno` は、データの完全性を視覚的に把握できるよう、便利なデータユーティリティをまとめたライブラリです。ここではこのライブラリのごく一部だけを紹介します。

In [ ]:
%matplotlib inline
msno.matrix(routes)

これにより、データを視覚的に素早く確認できます。右側のスパークラインはデータの完全性を示しています。次に airports を確認してみましょう。

In [ ]:
airports.info()

In [ ]:
msno.matrix(airports)

いいですね！全体として CSV はかなりきれいで、データも比較的よく揃っています。航空路線は約68,000件、空港は世界中で約7,000件あります。このリストが本物かどうか確かめるために、みんな大好きな空港であるラガーディア空港（LGA）を探してみましょう。

In [ ]:
airports[airports['IATA'] == 'LGA']

素晴らしいですね。では他のものも探してみましょう。世界で最も標高の高い空港、アルファベット順で最後に来る国の中で最も標高の高い空港、そしてオーストラリアで最も標高の高い空港です。

In [ ]:
highest = airports.sort_values('altitude', ascending=False).iloc[0]
highestInLastAlpha = airports.sort_values(['country', 'altitude'], ascending=[False, False]).iloc[0]
highestInAustralia = airports[airports['country'] == 'Australia'].sort_values('altitude', ascending=False).iloc[0]

In [ ]:
print(f'Highest airport: \n{highest}\n\n')
print(f'Highest airport in the country that comes last alphabetically: \n{highestInLastAlpha}\n\n')
print(f'Highest airport in Australia: \n{highestInAustralia}\n\n')

このデータでもっと実用的なことをしてみましょう。例えば、世界で最も混雑する路線トップ10を探します。

In [ ]:
busiest = routes.groupby(['source', 'dest']).count()['airline_id'].nlargest(10)
busiest

空港名をマージ（結合）することができます。

In [ ]:
busiest = pd.DataFrame(busiest).reset_index()
busiest.head()

In [ ]:
for sd in ['source', 'dest']:
    busiest = busiest.merge(airports[['IATA', 'name']].set_index('IATA'), how='left', left_on=sd, right_on='IATA')

busiest